<h1>Creating Random frames from the video</h1>

In [ ]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['clip.mp4'],
    total_images=100,
    out_dir="input_data",
    jpg_quality=100,
    seed=42
)

<h1>now converting coco json annotating data to YOLO Format</h1>

In [ ]:
import json
import os

# ---- CONFIG ----
COCO_PATH = "annotations/coco_annotations.json"
OUTPUT_DIR = "labels_all"
NUM_KP = 3
# ----------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(COCO_PATH, 'r') as f:
    coco = json.load(f)

# Build image id -> image info map
image_map = {img['id']: img for img in coco['images']}

saved = 0

for ann in coco['annotations']:
    img_info = image_map[ann['image_id']]
    img_w = img_info['width']
    img_h = img_info['height']
    stem  = os.path.splitext(img_info['file_name'])[0]

    # Bounding box normalized
    x, y, bw, bh = ann['bbox']
    cx = (x + bw / 2) / img_w
    cy = (y + bh / 2) / img_h
    nw = bw / img_w
    nh = bh / img_h

    # Keypoints normalized
    kps = ann['keypoints']
    kp_str = ""
    for i in range(NUM_KP):
        kx  = kps[i*3]     / img_w
        ky  = kps[i*3 + 1] / img_h
        vis = kps[i*3 + 2]
        kp_str += f" {kx:.6f} {ky:.6f} {vis}"

    line = f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}{kp_str}"

    with open(os.path.join(OUTPUT_DIR, stem + ".txt"), 'w') as f:
        f.write(line)

    saved += 1

print(f"Done! {saved} YOLO label files saved to {OUTPUT_DIR}/")

# Verify sample
sample = os.listdir(OUTPUT_DIR)[0]
with open(os.path.join(OUTPUT_DIR, sample)) as f:
    print(f"\nSample ({sample}):")
    print(f.read())

<h1>Now seperating training or testing validating data</h1>

In [ ]:
import os
import shutil
import random

# ---- CONFIG ----
FRAMES_FOLDER = "frames"
LABELS_FOLDER = "labels_all"
TRAIN_COUNT = 85
VAL_COUNT = 15
RANDOM_SEED = 42
# ----------------

random.seed(RANDOM_SEED)

# Get only frames that have a matching label (processed ones)
label_files = os.listdir(LABELS_FOLDER)
stems = [os.path.splitext(f)[0] for f in label_files]  # frame_0000, frame_0001 etc

print(f"Total labeled frames: {len(stems)}")

# Shuffle and split
random.shuffle(stems)
train_stems = stems[:TRAIN_COUNT]
val_stems   = stems[TRAIN_COUNT:TRAIN_COUNT + VAL_COUNT]

print(f"Train: {len(train_stems)} | Val: {len(val_stems)}")

# Create folders
for folder in [
    "dataset/train/images",
    "dataset/train/labels",
    "dataset/val/images",
    "dataset/val/labels"
]:
    os.makedirs(folder, exist_ok=True)

# Copy files
def copy_split(stems, split):
    missing_imgs = 0
    for stem in stems:
        # Copy image
        src_img = os.path.join(FRAMES_FOLDER, stem + ".jpg")
        dst_img = os.path.join(f"dataset/{split}/images", stem + ".jpg")
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        else:
            print(f"WARNING image not found: {src_img}")
            missing_imgs += 1

        # Copy label
        src_lbl = os.path.join(LABELS_FOLDER, stem + ".txt")
        dst_lbl = os.path.join(f"dataset/{split}/labels", stem + ".txt")
        shutil.copy(src_lbl, dst_lbl)

    print(f"{split} copied — missing images: {missing_imgs}")

copy_split(train_stems, "train")
copy_split(val_stems, "val")

print("\nDataset split complete!")
print(f"dataset/train/images → {len(os.listdir('dataset/train/images'))} files")
print(f"dataset/train/labels → {len(os.listdir('dataset/train/labels'))} files")
print(f"dataset/val/images   → {len(os.listdir('dataset/val/images'))} files")
print(f"dataset/val/labels   → {len(os.listdir('dataset/val/labels'))} files")

<h1>Train the data on YOLOb8 pose model give your annotaion data do it on colab with GPU</h1>

In [ ]:
from ultralytics import YOLO
from tqdm.notebook import tqdm
import time

model = YOLO('yolov8x-pose.pt')

EPOCHS = 100

# Custom callback to track progress
epoch_bar = tqdm(total=EPOCHS, desc="Training Progress", unit="epoch", colour="green")

current_epoch = [0]

def on_epoch_end(trainer):
    current_epoch[0] += 1
    ep = current_epoch[0]

    # Get latest metrics
    loss = trainer.loss.item() if hasattr(trainer.loss, 'item') else float(trainer.loss)

    epoch_bar.set_postfix({
        'epoch': f'{ep}/{EPOCHS}',
        'loss': f'{loss:.4f}',
        'remaining': f'{EPOCHS - ep} left'
    })
    epoch_bar.update(1)

model.add_callback('on_train_epoch_end', on_epoch_end)

results = model.train(
    data='bowling_pose.yaml',
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    patience=20,
    augment=True,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,
    project='bowling_runs',
    name='exp1',
    save=True,
    plots=True,
    device=0
)

epoch_bar.close()
print("Training complete!")
print("Best model saved at: bowling_runs/exp1/weights/best.pt")

<h1>Small test for validataing annotate points</h1>

In [ ]:
from ultralytics import YOLO

model = YOLO("weights/best.pt")

results = model.predict(source="clip.mp4", show=True, save=True)

In [4]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


<h1>Turning an ordinary cricket bowling video into an AI-powered visual biomechanics analysis video.</h1>

In [7]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import deque
from tqdm.notebook import tqdm
import math

# ---- CONFIG ----
MODEL_PATH = "weights/best.pt"
VIDEO_PATH = "clip.mp4"
OUTPUT_PATH = "output_analysis.mp4"

CONF_THRESHOLD = 0.3
TRAIL_LENGTH = 300
FAN_INTERVAL = 25
MAX_JUMP = 100
SMOOTH_WINDOW = 20
ARM_LENGTH_M = 2.5

GRAPH_W = 420
# ----------------

model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0:
    raise ValueError("Invalid video FPS.")

OUT_W = w + GRAPH_W

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (OUT_W, h))

if not out.isOpened():
    cap.release()
    raise RuntimeError(f"Cannot create output video: {OUTPUT_PATH}")

# ---- TRACKING BUFFERS ----

wrist_trail = deque(maxlen=TRAIL_LENGTH)
angle_history = deque(maxlen=200)
speed_history = deque(maxlen=200)
speed_buffer = deque(maxlen=7)

prev_wrist = None

frame_count = 0
detected_frames = 0

# ---- HELPER FUNCTIONS ----

def is_valid_jump(new_pt, prev_pt, max_jump):
    if prev_pt is None:
        return True

    dist = math.hypot(
        new_pt[0] - prev_pt[0],
        new_pt[1] - prev_pt[1]
    )

    return dist < max_jump


def smooth_trail(trail, window=20):
    pts = list(trail)

    if len(pts) < window:
        return pts

    smoothed = []

    for i in range(len(pts)):
        start = max(0, i - window // 2)
        end = min(len(pts), i + window // 2 + 1)

        wp = pts[start:end]

        avg_x = int(sum(p[0] for p in wp) / len(wp))
        avg_y = int(sum(p[1] for p in wp) / len(wp))

        smoothed.append((avg_x, avg_y))

    return smoothed


def get_trail_color(i, total):
    """
    Smooth multi-color trail:
    purple -> pink -> red -> orange -> yellow.

    OpenCV uses BGR color ordering.
    """
    ratio = i / max(total - 1, 1)

    color_stops = [
        (0.00, (180, 40, 145)),   # purple
        (0.25, (180, 55, 230)),   # pink
        (0.50, (30, 35, 245)),    # red
        (0.75, (0, 135, 255)),    # orange
        (1.00, (0, 235, 255))     # yellow
    ]

    for j in range(len(color_stops) - 1):
        start_pos, start_color = color_stops[j]
        end_pos, end_color = color_stops[j + 1]

        if start_pos <= ratio <= end_pos:
            t = (ratio - start_pos) / (end_pos - start_pos)

            return tuple(
                int(
                    start_color[channel]
                    + (end_color[channel] - start_color[channel]) * t
                )
                for channel in range(3)
            )

    return color_stops[-1][1]


def get_fan_color(i, total):
    ratio = i / max(total, 1)
    val = int(150 + ratio * 105)

    return (val, val, 255)


def calc_angle(shoulder, elbow, wrist):
    v1 = (
        shoulder[0] - elbow[0],
        shoulder[1] - elbow[1]
    )

    v2 = (
        wrist[0] - elbow[0],
        wrist[1] - elbow[1]
    )

    dot = v1[0] * v2[0] + v1[1] * v2[1]

    mag1 = math.hypot(v1[0], v1[1])
    mag2 = math.hypot(v2[0], v2[1])

    if mag1 * mag2 == 0:
        return None

    cosine = dot / (mag1 * mag2)
    cosine = max(-1, min(1, cosine))

    return math.degrees(math.acos(cosine))


def draw_graph(
    panel,
    history,
    title,
    unit,
    color,
    y_offset,
    max_val,
    graph_h=160
):
    gx, gy = 25, y_offset
    gw = GRAPH_W - 50
    gh = max(20, graph_h)

    cv2.rectangle(
        panel,
        (gx - 8, gy - 35),
        (gx + gw + 8, gy + gh + 15),
        (25, 25, 25),
        -1
    )

    cv2.rectangle(
        panel,
        (gx - 8, gy - 35),
        (gx + gw + 8, gy + gh + 15),
        (80, 80, 80),
        1
    )

    cv2.putText(
        panel,
        title,
        (gx, gy - 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (220, 220, 220),
        2
    )

    # Grid lines and labels
    for i in range(5):
        y_grid = gy + int(gh * i / 4)

        cv2.line(
            panel,
            (gx, y_grid),
            (gx + gw, y_grid),
            (45, 45, 45),
            1
        )

        val_label = int(max_val - max_val * i / 4)

        cv2.putText(
            panel,
            str(val_label),
            (gx - 5, y_grid + 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (100, 100, 100),
            1
        )

    pts = list(history)

    if len(pts) > 1:
        poly_pts = []

        for i, val in enumerate(pts):
            if val is None:
                continue

            x = gx + int(i / max(len(pts) - 1, 1) * gw)

            y = gy + gh - int(
                min(max(val, 0), max_val) / max_val * gh
            )

            poly_pts.append((x, y))

        if len(poly_pts) > 1:
            fill_pts = (
                [(poly_pts[0][0], gy + gh)]
                + poly_pts
                + [(poly_pts[-1][0], gy + gh)]
            )

            fill_arr = np.array(fill_pts, dtype=np.int32)

            overlay = panel.copy()

            cv2.fillPoly(overlay, [fill_arr], color)

            cv2.addWeighted(
                overlay,
                0.2,
                panel,
                0.8,
                0,
                panel
            )

            for i in range(1, len(poly_pts)):
                cv2.line(
                    panel,
                    poly_pts[i - 1],
                    poly_pts[i],
                    color,
                    2
                )

    # Current value
    if pts and pts[-1] is not None:
        cv2.putText(
            panel,
            f"{pts[-1]:.1f} {unit}",
            (gx + gw - 120, gy - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )


# ---- MAIN VIDEO PROCESSING LOOP ----

progress = tqdm(
    total=total_frames,
    desc="Inference",
    unit="frame",
    colour="green"
)

try:
    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        # Create analytics panel
        panel = np.zeros((h, GRAPH_W, 3), dtype=np.uint8)
        panel[:] = (15, 15, 15)

        results = model(frame, verbose=False)

        detected = False

        shoulder_pt = None
        elbow_pt = None
        wrist_pt = None

        # ---- KEYPOINT DETECTION ----

        for result in results:

            if (
                result.keypoints is not None
                and len(result.keypoints.data) > 0
            ):

                kps = result.keypoints.data[0]

                pts = {}

                for kp_idx in range(3):

                    if kp_idx < len(kps):

                        x, y, conf = kps[kp_idx]

                        x = int(x.item())
                        y = int(y.item())
                        conf = conf.item()

                        if (
                            conf > CONF_THRESHOLD
                            and x > 0
                            and y > 0
                        ):
                            pts[kp_idx] = (x, y)

                if pts:

                    detected = True

                    shoulder_pt = pts.get(0)
                    elbow_pt = pts.get(1)
                    wrist_pt = pts.get(2)

                    # ---- WRIST TRAIL ----

                    if wrist_pt:

                        last = wrist_trail[-1] if wrist_trail else None

                        if is_valid_jump(wrist_pt, last, MAX_JUMP):
                            wrist_trail.append(wrist_pt)

                    # ---- ARM LINES ----

                    if shoulder_pt and elbow_pt:
                        cv2.line(
                            frame,
                            shoulder_pt,
                            elbow_pt,
                            (255, 255, 255),
                            2
                        )

                    if elbow_pt and wrist_pt:
                        cv2.line(
                            frame,
                            elbow_pt,
                            wrist_pt,
                            (255, 255, 255),
                            2
                        )

                    # ---- KEYPOINT MARKERS ----

                    colors = {
                        0: (100, 100, 255),
                        1: (0, 165, 255),
                        2: (0, 255, 100)
                    }

                    names = {
                        0: "S",
                        1: "E",
                        2: "W"
                    }

                    for kp_idx, pt in pts.items():

                        cv2.circle(
                            frame,
                            pt,
                            9,
                            colors[kp_idx],
                            -1
                        )

                        cv2.circle(
                            frame,
                            pt,
                            11,
                            (255, 255, 255),
                            1
                        )

                        cv2.putText(
                            frame,
                            names[kp_idx],
                            (pt[0] + 12, pt[1] - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.55,
                            colors[kp_idx],
                            2
                        )

                    # ---- ELBOW ANGLE ----

                    if shoulder_pt and elbow_pt and wrist_pt:

                        angle = calc_angle(
                            shoulder_pt,
                            elbow_pt,
                            wrist_pt
                        )

                        if angle is not None:

                            angle_history.append(angle)

                            cv2.putText(
                                frame,
                                f"{angle:.1f}",
                                (elbow_pt[0] + 15, elbow_pt[1] - 15),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.7,
                                (0, 255, 255),
                                2
                            )

                    # ---- PIXEL TO METER SCALE ----

                    px_to_m = None

                    if shoulder_pt and wrist_pt:

                        arm_px = math.hypot(
                            wrist_pt[0] - shoulder_pt[0],
                            wrist_pt[1] - shoulder_pt[1]
                        )

                        if arm_px > 0:
                            px_to_m = ARM_LENGTH_M / arm_px

                    # ---- WRIST SPEED ----

                    if wrist_pt and prev_wrist and px_to_m:

                        if is_valid_jump(
                            wrist_pt,
                            prev_wrist,
                            MAX_JUMP
                        ):

                            dx = wrist_pt[0] - prev_wrist[0]
                            dy = wrist_pt[1] - prev_wrist[1]

                            speed_px = math.hypot(dx, dy) * fps

                            speed_ms = speed_px * px_to_m

                            speed_buffer.append(speed_ms)

                            smoothed_speed = (
                                sum(speed_buffer) / len(speed_buffer)
                            )

                            speed_history.append(smoothed_speed)

                    if wrist_pt:
                        prev_wrist = wrist_pt

                    break

        # Reset wrist reference when detection is lost
        if not detected:
            prev_wrist = None

        # ---- SMOOTHED WRIST TRAIL ----

        smoothed = smooth_trail(
            wrist_trail,
            SMOOTH_WINDOW
        )

        # Draw a persistent, anti-aliased, multi-color wrist trail.
        # The deque retains the last TRAIL_LENGTH valid wrist points,
        # so the trail remains visible as the bowler moves.
        for i in range(1, len(smoothed)):

            color = get_trail_color(i, len(smoothed))

            # Gradually increase line thickness toward the newest point.
            thickness = max(
                2,
                int(6 * i / max(len(smoothed) - 1, 1))
            )

            cv2.line(
                frame,
                smoothed[i - 1],
                smoothed[i],
                color,
                thickness,
                lineType=cv2.LINE_AA
            )

            if i % 8 == 0:
                cv2.circle(
                    frame,
                    smoothed[i],
                    4,
                    (255, 255, 255),
                    -1,
                    lineType=cv2.LINE_AA
                )

        # ---- FAN LINES ----

        if elbow_pt and len(smoothed) >= FAN_INTERVAL:

            sampled = smoothed[::FAN_INTERVAL]

            for i, wp in enumerate(sampled):

                dist = math.hypot(
                    wp[0] - elbow_pt[0],
                    wp[1] - elbow_pt[1]
                )

                if dist < 500:

                    color = get_fan_color(
                        i,
                        len(sampled)
                    )

                    cv2.line(
                        frame,
                        elbow_pt,
                        wp,
                        color,
                        1
                    )

        # ---- ANALYTICS PANEL ----

        cv2.putText(
            panel,
            "BOWLING ANALYSIS",
            (15, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 200, 255),
            2
        )

        cv2.line(
            panel,
            (15, 45),
            (GRAPH_W - 15, 45),
            (60, 60, 60),
            1
        )

        draw_graph(
            panel,
            angle_history,
            "Elbow Angle",
            "deg",
            (0, 255, 255),
            75,
            180,
            graph_h=int(h // 2 - 120)
        )

        draw_graph(
            panel,
            speed_history,
            "Wrist Speed",
            "m/s",
            (0, 255, 100),
            h // 2 + 20,
            20,
            graph_h=int(h // 2 - 120)
        )

        cv2.putText(
            panel,
            f"Frame {frame_count}/{total_frames}",
            (15, h - 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (120, 120, 120),
            1
        )

        # ---- COMBINE FRAMES ----

        combined = np.hstack([frame, panel])

        out.write(combined)

        if detected:
            detected_frames += 1

        frame_count += 1

        progress.set_postfix({
            "left": total_frames - frame_count,
            "detected": f"{detected_frames}/{frame_count}"
        })

        progress.update(1)

finally:
    cap.release()
    out.release()
    progress.close()

# ---- FINAL REPORT ----

print("\nDone!")

if frame_count > 0:
    print(
        f"Detection rate: "
        f"{detected_frames / frame_count * 100:.1f}%"
    )

print(f"Processed frames: {frame_count}")
print(f"Output saved: {OUTPUT_PATH}")

Inference:   0%|          | 0/1466 [00:00<?, ?frame/s]


Done!
Detection rate: 97.8%
Processed frames: 1466
Output saved: output_analysis.mp4
